In [ ]:
import ee
import geemap
from utils.variables import (
    PROJECT,
    TEST_SITE_IDs,
    ANALYSIS_START_YR,
    ANALYSIS_END_YR,
    GLC_CLASSES
)

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Data imports
PAs = (ee.FeatureCollection("WCMC/WDPA/current/polygons")
       .filter(ee.Filter.eq("REALM", "Terrestrial")))
OECMs = (ee.FeatureCollection("WCMC/WDOECM/current/polygons")
         .filter(ee.Filter.eq("REALM", "Terrestrial")))
GLC = (ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
       .mosaic())
HGFC = (ee.Image("UMD/hansen/global_forest_change_2024_v1_12"))

In [ ]:
# Select test sites
all_sites = (ee.FeatureCollection([PAs, OECMs]).flatten())
test_sites = (all_sites.filter(ee.Filter.inList("SITE_ID", TEST_SITE_IDs)))

In [ ]:
# Process Global Land Cover Change data

# Select bands corresponding to analysis years
analysis_years = list[int](range(ANALYSIS_START_YR, ANALYSIS_END_YR + 1))
band_names = [f"b{year - 2000 + 1}" for year in analysis_years]
new_band_names = [f"GLC_{year}" for year in analysis_years]
GLC_filtered = GLC.select(band_names, new_band_names)

# Remap class values to 1-36
def remap_classes(band):
    return (GLC_filtered.select(band)
            .remap(GLC_CLASSES, ee.List.sequence(1, len(GLC_CLASSES)), defaultValue=0)
            .rename([band]))

remapped_bands = [remap_classes(band) for band in new_band_names]
GLC_remapped = ee.Image.cat(remapped_bands)

In [ ]:
# Process Hansen Global Forest Change data

forest_loss = HGFC.select("lossyear")
mask = (forest_loss.gte(ANALYSIS_START_YR - 2000)
        .And(forest_loss.lte(ANALYSIS_END_YR - 2000)))
forest_loss_masked = forest_loss.updateMask(mask)

In [ ]:
# Visualization
from utils.variables import GLC_PALETTE

Map = geemap.Map()
Map.addLayer(GLC_remapped.select('GLC_2022'), {'min':0, 'max':36, 'palette': GLC_PALETTE}, "GLC")
Map.addLayer(forest_loss_masked, {'min':ANALYSIS_START_YR-2000, 'max':ANALYSIS_END_YR-2000, 'palette': ['yellow', 'red']}, "Forest loss")
Map.addLayer(test_sites, {"color": "red"}, "Test sites")
Map